# 04.8 · Pipelines and ColumnTransformer: leak-proof preprocessing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/08-pipelines/lesson.ipynb)

Read the [README](README.md) first and do the four-number scaling example on paper. This notebook repeats it, shows two leaks (one harmless-looking, one dramatic), then builds the complete ShopWave churn pipeline that takes raw customer rows in and gives probabilities out.

## 1. Scaling by hand: fit on train only vs fit on everything

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

train_vals = np.array([10, 20, 30])
test_val = 100

mu, sd = train_vals.mean(), train_vals.std()
print(f"fit on train only: mean {mu:.0f}, std {sd:.3f}  →  test z = {(test_val - mu) / sd:.2f}")

everything = np.append(train_vals, test_val)
mu2, sd2 = everything.mean(), everything.std()
print(f"fit on everything: mean {mu2:.0f}, std {sd2:.3f} →  test z = {(test_val - mu2) / sd2:.2f}")

fit on train only: mean 20, std 8.165  →  test z = 9.80
fit on everything: mean 40, std 35.355 →  test z = 1.70


Fitted honestly, the test customer is a startling 9.8 standard deviations out: something the model has never seen. Fitted on everything, the scaler has already "seen" them and they look ordinary at 1.7. The test row changed the preprocessing, so the test is no longer a test.

## 2. The real data

In [2]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

cust["avg_basket"] = cust["spend"] / cust["orders"]
cust["orders_per_month"] = cust["orders"] / (cust["tenure_days"] / 30).clip(lower=1)
cust["log_spend"] = np.log1p(cust["spend"])
country = past.groupby("Customer ID")["Country"].agg(lambda s: s.mode()[0])
cust["country"] = country.reindex(cust.index)
cust["is_uk"] = (cust["country"] == "United Kingdom").astype(int)
weekend = past.assign(wk=past["InvoiceDate"].dt.dayofweek >= 5).groupby("Customer ID")["wk"].mean()
cust["share_weekend"] = weekend.reindex(cust.index)
autumn = past[past["InvoiceDate"] >= "2011-06-01"].groupby("Customer ID")["Invoice"].nunique()
cust["orders_last_3m"] = autumn.reindex(cust.index).fillna(0)

from sklearn.model_selection import train_test_split
num_cols = ["orders", "spend", "products", "tenure_days", "days_since_last", "avg_basket"]
cat_cols = ["country"]
train, test = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
X_train, y_train = train[num_cols + cat_cols], train["churned"]
X_test, y_test = test[num_cols + cat_cols], test["churned"]
print(f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%


train 2,487 / test 830


## 3. Leak 1: scale first, then cross-validate

The tempting order: scale the whole training set, then run cross-validation. Each held-out fold has then influenced the mean and std used to scale it.

In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
skf = StratifiedKFold(5, shuffle=True, random_state=42)
Xn = X_train[num_cols]

leaky = cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(Xn), y_train, cv=skf)
honest = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), Xn, y_train, cv=skf)
print(f"scale then CV (leaky):   {leaky.mean():.4f}")
print(f"pipeline inside CV:      {honest.mean():.4f}")

scale then CV (leaky):   0.6856
pipeline inside CV:      0.6856


Identical to four decimal places (0.6856). With 2,000 rows, a mean and a standard deviation barely change when one fold is left out. So is the "rule" just pedantry? Look at the next leak.

## 4. Leak 2: choose features first, then cross-validate

Pure noise: 200 customers, 5,000 random columns, random labels. There is **nothing** to learn; honest accuracy must be about 50%. We pick the 20 columns most correlated with the label, then cross-validate.

In [4]:
from sklearn.feature_selection import SelectKBest, f_classif
rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 5000))
y_noise = rng.integers(0, 2, 200)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

picked = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)      # uses ALL rows
leaky = cross_val_score(LogisticRegression(max_iter=1000), picked, y_noise, cv=cv).mean()
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)),
                         X_noise, y_noise, cv=cv).mean()
print(f"select then CV (leaky): {leaky:.3f}")
print(f"pipeline inside CV:     {honest:.3f}")

select then CV (leaky): 0.795
pipeline inside CV:     0.550


The leaky version claims about **80%** accuracy on random noise. The selection step looked at every row's label and kept columns that happened to match those labels, including the rows later used for testing. Inside a pipeline, selection is redone in each fold using only the training rows, and the honest answer (about 55%, i.e. chance) appears.

The rule is the same for both leaks: **anything that learns from data must be fitted inside the fold.** A pipeline guarantees it.

## 5. Building the ShopWave pipeline

Numeric columns: fill missing values with the median, log-transform, scale. Categorical column (`country`): fill missing with the most common value, one-hot encode, lump rare countries (fewer than 20 customers) into one "infrequent" column.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression

numeric = make_pipeline(SimpleImputer(strategy="median"),
                        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                        StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                                          sparse_output=False))
prep = ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)])
pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

## 6. Cross-validate the whole thing, then test once

In [6]:
from sklearn.metrics import roc_auc_score
s = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="roc_auc")
print("CV AUC per fold:", s.round(3), f" mean {s.mean():.3f} ± {s.std():.3f}")

pipe.fit(X_train, y_train)
print(f"test AUC: {roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]):.3f}")

CV AUC per fold: [0.707 0.766 0.704 0.752 0.762]  mean 0.738 ± 0.027
test AUC: 0.746


## 7. Look inside: `set_output(transform="pandas")`

By default a ColumnTransformer returns a bare NumPy array. Ask for a DataFrame to see exactly what the model receives.

In [7]:
pipe.set_output(transform="pandas")
pipe.fit(X_train, y_train)
seen = pipe.named_steps["prep"].transform(X_test.head(3))
seen.round(2)

,num__orders,num__spend,num__products,num__tenure_days,num__days_since_last,num__avg_basket,cat__country_France,cat__country_Germany,cat__country_United Kingdom,cat__country_infrequent_sklearn
Customer ID,,,,,,,,,,
14915.0,-0.86,-1.16,-1.05,0.25,0.97,-0.83,0.0,0.0,1.0,0.0
14193.0,-0.86,-0.27,-0.16,0.19,0.93,0.57,0.0,0.0,1.0,0.0
16681.0,0.63,0.67,-0.53,0.06,-0.18,0.27,0.0,0.0,1.0,0.0


In [8]:
print(len(seen.columns), "columns go into the model:")
print(seen.columns.tolist())

10 columns go into the model:
['num__orders', 'num__spend', 'num__products', 'num__tenure_days', 'num__days_since_last', 'num__avg_basket', 'cat__country_France', 'cat__country_Germany', 'cat__country_United Kingdom', 'cat__country_infrequent_sklearn']


Six numeric columns (logged and scaled), one column each for the three countries with at least 20 training customers (France, Germany, United Kingdom), and `country_infrequent_sklearn` for the other 30 countries.

## 8. Raw rows in, probabilities out

The pipeline remembers every step: medians, logs, means, standard deviations, the list of countries. New customers can arrive messy:

In [9]:
new_customers = pd.DataFrame({
    "orders": [1, 12], "spend": [80.0, np.nan], "products": [3, 140],
    "tenure_days": [20, 260], "days_since_last": [20, 5], "avg_basket": [80.0, np.nan],
    "country": ["Kenya", "Germany"],          # Kenya never appeared in training
})
new_customers["p_churn"] = pipe.predict_proba(new_customers)[:, 1].round(3)
new_customers

,orders,spend,products,tenure_days,days_since_last,avg_basket,country,p_churn
0,1,80.0,3,20,20,80.0,Kenya,0.720
1,12,NaN,140,260,5,NaN,Germany,0.027


The missing spend was filled with the training median, the unseen country went into the "infrequent" bucket, and nothing crashed. The first customer (one order, new) is likely to churn; the second (12 orders, active five days ago) is not.

This one object is what you will save and deploy in Module 10.

## Quiz answers

1. Every step that *learns* something from data (imputer medians, scaler means, encoder categories, feature selection) is fitted on the training part of each fold only, then applied to the held-out part.
2. `ColumnTransformer` sends different columns through different preprocessing (numbers to impute + scale, text categories to one-hot) and glues the results side by side.
3. Fitting the scaler on all rows used information from the rows later used for testing; for a mean and std on 2,000 rows that information is tiny.
4. Selecting features on all rows lets the labels of the test rows choose the features. On pure noise this pushed CV accuracy from about 55% to about 80%.
5. `handle_unknown="ignore"` or `"infrequent_if_exist"` on the `OneHotEncoder`; and an imputer for missing values. Otherwise `predict` raises an error on the new value.

**Next:** [04.9 Bias and variance](../09-bias-variance/)